# Improved Spectral Subtraction for Milling Process Analysis

This notebook implements several robust spectral subtraction methods:
- **Power Spectral Subtraction**: Classic method with over-subtraction factor
- **Magnitude Spectral Subtraction**: Works directly on magnitude with half-wave rectification
- **Multi-band Subtraction**: Frequency-dependent subtraction for better control

Key improvements:
- Proper noise profile estimation (mean over time)
- Over-subtraction factor for aggressive noise removal
- Spectral flooring to prevent musical artifacts
- No normalization (preserves absolute levels)
- Comprehensive before/after visualization

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy import signal

# Larger figure sizes for better visibility
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (14, 5)

## Method 1: Power Spectral Subtraction (Recommended)

This is the most effective method for constant noise. It works on power spectra with proper over-subtraction.

In [ ]:
def spectral_subtract_power(mix_path, noise_path, out_path=None,
                            n_fft=4096, hop_length=512,
                            alpha=2.0, beta=0.01):
    """
    Power Spectral Subtraction - Most effective for stationary noise
    
    Parameters:
    -----------
    mix_path : str
        Audio with air + milling process
    noise_path : str  
        Air-only reference audio
    alpha : float (default 2.0)
        Over-subtraction factor. Higher = more aggressive noise removal.
        Typical range: 1.5 - 3.0
        - 1.0 = basic subtraction (often insufficient)
        - 2.0 = recommended for constant noise
        - 3.0 = very aggressive (may introduce artifacts)
    beta : float (default 0.01)
        Spectral floor factor (fraction of original signal power to preserve).
        Prevents over-subtraction and musical noise.
        Typical range: 0.001 - 0.05
    
    Returns:
    --------
    y_clean : array
        Cleaned audio signal
    sr : int
        Sample rate
    X_mag : array
        Original magnitude spectrogram
    clean_mag : array
        Cleaned magnitude spectrogram
    noise_profile : array
        Estimated noise power profile
    """
    
    # Load audio WITHOUT normalization to preserve absolute levels
    mix, sr = sf.read(mix_path, always_2d=False)
    noise, sr_noise = sf.read(noise_path, always_2d=False)
    
    if sr != sr_noise:
        raise ValueError(f"Sample rate mismatch: {sr} vs {sr_noise}")
    
    # Convert to mono if stereo
    if mix.ndim > 1:
        mix = mix.mean(axis=1)
    if noise.ndim > 1:
        noise = noise.mean(axis=1)
    
    print(f"Mix signal: {len(mix)/sr:.2f}s, peak={np.max(np.abs(mix)):.4f}")
    print(f"Noise signal: {len(noise)/sr:.2f}s, peak={np.max(np.abs(noise)):.4f}")
    
    # Compute STFT
    X = librosa.stft(mix, n_fft=n_fft, hop_length=hop_length)
    N = librosa.stft(noise, n_fft=n_fft, hop_length=hop_length)
    
    # Get magnitude and phase
    X_mag = np.abs(X)
    X_phase = np.angle(X)
    N_mag = np.abs(N)
    
    # Estimate noise POWER profile as mean over time
    # This is correct for stationary noise - we want the average noise characteristics
    noise_power_profile = np.mean(N_mag**2, axis=1, keepdims=True)
    
    print(f"Noise profile shape: {noise_power_profile.shape}")
    print(f"Noise power range: {np.min(noise_power_profile):.2e} to {np.max(noise_power_profile):.2e}")
    
    # Power spectral subtraction with over-subtraction
    X_power = X_mag**2
    clean_power = X_power - alpha * noise_power_profile
    
    # Apply spectral floor (as fraction of original power)
    floor = beta * X_power
    clean_power = np.maximum(clean_power, floor)
    
    # Convert back to magnitude
    clean_mag = np.sqrt(clean_power)
    
    # Reconstruct with original phase (phase is not affected by stationary noise)
    Y = clean_mag * np.exp(1j * X_phase)
    y_clean = librosa.istft(Y, hop_length=hop_length, length=len(mix))
    
    print(f"Clean signal: peak={np.max(np.abs(y_clean)):.4f}")
    print(f"Power reduction: {10*np.log10(np.mean(y_clean**2) / (np.mean(mix**2) + 1e-10)):.2f} dB")
    
    # Save if requested
    if out_path:
        sf.write(out_path, y_clean, sr)
        print(f"Saved to: {out_path}")
    
    return y_clean, sr, X_mag, clean_mag, noise_power_profile

## Method 2: Magnitude Spectral Subtraction (Alternative)

Works directly on magnitude spectra. Can be gentler but less effective for strong noise.

In [ ]:
def spectral_subtract_magnitude(mix_path, noise_path, out_path=None,
                                n_fft=4096, hop_length=512,
                                alpha=1.5, beta=0.02):
    """
    Magnitude Spectral Subtraction - Alternative method
    
    Similar parameters to power method but works on magnitude directly.
    Can be more conservative and produce fewer artifacts.
    """
    
    # Load audio
    mix, sr = sf.read(mix_path, always_2d=False)
    noise, sr_noise = sf.read(noise_path, always_2d=False)
    
    if sr != sr_noise:
        raise ValueError(f"Sample rate mismatch: {sr} vs {sr_noise}")
    
    if mix.ndim > 1:
        mix = mix.mean(axis=1)
    if noise.ndim > 1:
        noise = noise.mean(axis=1)
    
    # STFT
    X = librosa.stft(mix, n_fft=n_fft, hop_length=hop_length)
    N = librosa.stft(noise, n_fft=n_fft, hop_length=hop_length)
    
    X_mag = np.abs(X)
    X_phase = np.angle(X)
    N_mag = np.abs(N)
    
    # Noise magnitude profile
    noise_mag_profile = np.mean(N_mag, axis=1, keepdims=True)
    
    # Magnitude subtraction with half-wave rectification
    clean_mag = X_mag - alpha * noise_mag_profile
    
    # Apply floor and ensure non-negative
    floor = beta * X_mag
    clean_mag = np.maximum(clean_mag, floor)
    
    # Reconstruct
    Y = clean_mag * np.exp(1j * X_phase)
    y_clean = librosa.istft(Y, hop_length=hop_length, length=len(mix))
    
    if out_path:
        sf.write(out_path, y_clean, sr)
    
    return y_clean, sr, X_mag, clean_mag, noise_mag_profile

## Method 3: Multi-band Spectral Subtraction (Advanced)

Applies different subtraction factors per frequency band. Useful if noise characteristics vary with frequency.

In [ ]:
def spectral_subtract_multiband(mix_path, noise_path, out_path=None,
                                n_fft=4096, hop_length=512,
                                alpha_low=2.5, alpha_mid=2.0, alpha_high=1.5,
                                beta=0.01):
    """
    Multi-band Spectral Subtraction - Frequency-dependent processing
    
    Parameters:
    -----------
    alpha_low : float
        Over-subtraction for low frequencies (0-500 Hz)
    alpha_mid : float  
        Over-subtraction for mid frequencies (500-2000 Hz)
    alpha_high : float
        Over-subtraction for high frequencies (2000+ Hz)
        
    Note: Air noise is often stronger in low frequencies, so alpha_low should be highest.
    """
    
    # Load audio
    mix, sr = sf.read(mix_path, always_2d=False)
    noise, sr_noise = sf.read(noise_path, always_2d=False)
    
    if sr != sr_noise:
        raise ValueError(f"Sample rate mismatch: {sr} vs {sr_noise}")
    
    if mix.ndim > 1:
        mix = mix.mean(axis=1)
    if noise.ndim > 1:
        noise = noise.mean(axis=1)
    
    # STFT
    X = librosa.stft(mix, n_fft=n_fft, hop_length=hop_length)
    N = librosa.stft(noise, n_fft=n_fft, hop_length=hop_length)
    
    X_mag = np.abs(X)
    X_phase = np.angle(X)
    N_mag = np.abs(N)
    
    # Noise power profile
    noise_power_profile = np.mean(N_mag**2, axis=1, keepdims=True)
    
    # Create frequency-dependent alpha
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    alpha = np.ones_like(freqs) * alpha_mid
    alpha[freqs < 500] = alpha_low
    alpha[freqs > 2000] = alpha_high
    alpha = alpha.reshape(-1, 1)  # Shape for broadcasting
    
    # Power subtraction with frequency-dependent alpha
    X_power = X_mag**2
    clean_power = X_power - alpha * noise_power_profile
    
    # Floor
    floor = beta * X_power
    clean_power = np.maximum(clean_power, floor)
    
    clean_mag = np.sqrt(clean_power)
    
    # Reconstruct
    Y = clean_mag * np.exp(1j * X_phase)
    y_clean = librosa.istft(Y, hop_length=hop_length, length=len(mix))
    
    if out_path:
        sf.write(out_path, y_clean, sr)
    
    return y_clean, sr, X_mag, clean_mag, noise_power_profile

## Visualization Functions

Comprehensive before/after comparison to evaluate effectiveness

In [ ]:
def plot_spectral_comparison(mix_path, noise_path, y_clean, sr, X_mag, clean_mag,
                            noise_profile, n_fft=4096, hop_length=512):
    """
    Create comprehensive visualization of spectral subtraction results
    """
    
    # Load original for comparison
    mix, _ = sf.read(mix_path, always_2d=False)
    if mix.ndim > 1:
        mix = mix.mean(axis=1)
    
    # Compute power spectrograms for better visualization
    X_power_db = librosa.amplitude_to_db(X_mag, ref=np.max)
    clean_power_db = librosa.amplitude_to_db(clean_mag, ref=np.max)
    
    # Create figure with multiple subplots
    fig = plt.figure(figsize=(16, 12))
    
    # 1. Waveform comparison
    ax1 = plt.subplot(4, 2, 1)
    librosa.display.waveshow(mix, sr=sr, ax=ax1, alpha=0.7, label='Original')
    ax1.set_title('Original Mix Waveform (Air + Process)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)
    
    ax2 = plt.subplot(4, 2, 2)
    librosa.display.waveshow(y_clean, sr=sr, ax=ax2, alpha=0.7, label='Cleaned', color='green')
    ax2.set_title('Cleaned Signal (Process Only)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Amplitude')
    ax2.grid(True, alpha=0.3)
    
    # 2. Full-band spectrogram comparison
    ax3 = plt.subplot(4, 2, 3)
    img3 = librosa.display.specshow(X_power_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=ax3, cmap='viridis')
    ax3.set_title('Original Spectrogram', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Frequency (Hz)')
    plt.colorbar(img3, ax=ax3, format='%+2.0f dB')
    
    ax4 = plt.subplot(4, 2, 4)
    img4 = librosa.display.specshow(clean_power_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=ax4, cmap='viridis')
    ax4.set_title('Cleaned Spectrogram', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Frequency (Hz)')
    plt.colorbar(img4, ax=ax4, format='%+2.0f dB')
    
    # 3. Zoomed spectrogram (0-5000 Hz) for process detail
    ax5 = plt.subplot(4, 2, 5)
    img5 = librosa.display.specshow(X_power_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=ax5, cmap='viridis')
    ax5.set_ylim([2500, 7500])
    ax5.set_title('Original Spectrogram (2.5-7.5 kHz zoom)', fontsize=12, fontweight='bold')
    ax5.set_ylabel('Frequency (Hz)')
    plt.colorbar(img5, ax=ax5, format='%+2.0f dB')
    
    ax6 = plt.subplot(4, 2, 6)
    img6 = librosa.display.specshow(clean_power_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=ax6, cmap='viridis')
    ax6.set_ylim([2500, 7500])
    ax6.set_title('Cleaned Spectrogram (2.5-7.5 kHz zoom)', fontsize=12, fontweight='bold')
    ax6.set_ylabel('Frequency (Hz)')
    plt.colorbar(img6, ax=ax6, format='%+2.0f dB')
    
    # 4. Difference spectrogram (what was removed)
    ax7 = plt.subplot(4, 2, 7)
    difference_db = X_power_db - clean_power_db
    img7 = librosa.display.specshow(difference_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=ax7, cmap='RdBu_r')
    ax7.set_title('Removed Energy (Original - Cleaned)', fontsize=12, fontweight='bold')
    ax7.set_ylabel('Frequency (Hz)')
    ax7.set_xlabel('Time (s)')
    plt.colorbar(img7, ax=ax7, format='%+2.0f dB', label='Removed dB')
    
    # 5. Noise profile plot
    ax8 = plt.subplot(4, 2, 8)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    noise_db = 10 * np.log10(noise_profile.flatten() + 1e-10)
    ax8.plot(freqs, noise_db, linewidth=2, color='red', label='Estimated Noise Profile')
    ax8.set_xlabel('Frequency (Hz)')
    ax8.set_ylabel('Power (dB)')
    ax8.set_title('Noise Power Profile', fontsize=12, fontweight='bold')
    ax8.grid(True, alpha=0.3)
    ax8.set_xlim([0, sr/2])
    ax8.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n" + "="*60)
    print("SPECTRAL SUBTRACTION ANALYSIS")
    print("="*60)
    print(f"Original RMS:  {np.sqrt(np.mean(mix**2)):.6f}")
    print(f"Cleaned RMS:   {np.sqrt(np.mean(y_clean**2)):.6f}")
    print(f"RMS Reduction: {20*np.log10(np.sqrt(np.mean(y_clean**2)) / (np.sqrt(np.mean(mix**2)) + 1e-10)):.2f} dB")
    print(f"\nOriginal Peak: {np.max(np.abs(mix)):.6f}")
    print(f"Cleaned Peak:  {np.max(np.abs(y_clean)):.6f}")
    
    # Frequency band analysis
    print("\nEnergy by Frequency Band:")
    bands = [(0, 500), (500, 1000), (1000, 2000), (2000, 5000), (5000, sr/2)]
    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        if np.any(mask):
            orig_energy = np.mean(X_mag[mask, :]**2)
            clean_energy = np.mean(clean_mag[mask, :]**2)
            reduction = 10*np.log10(clean_energy / (orig_energy + 1e-10))
            print(f"  {lo:>5.0f}-{hi:>5.0f} Hz: {reduction:>6.2f} dB reduction")
    print("="*60)

## Usage Example

Run the spectral subtraction and visualize results

In [ ]:
# Set your file paths
mix_path = "../data/raw_data_nuten/4mm_mit_Luft_front_1.flac"      # Air + milling process
noise_path = "../data/raw_data_nuten/only_air_front_1.flac"      # Air only (reference)

### Try Method 1: Power Spectral Subtraction (Recommended)

Start with alpha=2.0. If noise remains, increase to 2.5 or 3.0. If artifacts appear, decrease to 1.5.

In [ ]:
# Run power spectral subtraction
y_clean, sr, X_mag, clean_mag, noise_profile = spectral_subtract_power(
    mix_path, noise_path,
    out_path="cleaned_power.flac",
    n_fft=4096,
    hop_length=512,
    alpha=3,      # Increase if noise remains (try 2.5, 3.0)
    beta=0.05       # Increase if musical artifacts appear (try 0.02, 0.05)
)

# Visualize results
plot_spectral_comparison(mix_path, noise_path, y_clean, sr, X_mag, clean_mag,
                        noise_profile, n_fft=4096, hop_length=512)

### Try Method 2: Magnitude Spectral Subtraction (If Method 1 has too many artifacts)

In [ ]:
# Run magnitude spectral subtraction
y_clean2, sr2, X_mag2, clean_mag2, noise_profile2 = spectral_subtract_magnitude(
    mix_path, noise_path,
    out_path="cleaned_magnitude.flac",
    n_fft=4096,
    hop_length=512,
    alpha=1.5,
    beta=0.02
)

plot_spectral_comparison(mix_path, noise_path, y_clean2, sr2, X_mag2, clean_mag2,
                        noise_profile2, n_fft=4096, hop_length=512)

### Try Method 3: Multi-band Subtraction (If noise varies with frequency)

In [ ]:
# Run multi-band spectral subtraction
y_clean3, sr3, X_mag3, clean_mag3, noise_profile3 = spectral_subtract_multiband(
    mix_path, noise_path,
    out_path="cleaned_multiband.flac",
    n_fft=4096,
    hop_length=512,
    alpha_low=2.5,   # Low frequencies often have more noise
    alpha_mid=2.0,
    alpha_high=1.5,  # High frequencies may need gentler treatment
    beta=0.01
)

plot_spectral_comparison(mix_path, noise_path, y_clean3, sr3, X_mag3, clean_mag3,
                        noise_profile3, n_fft=4096, hop_length=512)

## Additional Analysis: Time-Frequency Detail

Create high-resolution spectrograms focused on the process-relevant frequency range

In [ ]:
# Create detailed spectrograms for process analysis
def plot_detailed_spectrograms(y_clean, sr, fmax=8000):
    """
    High-resolution spectrograms for detecting process variations
    """
    # Use shorter window for better time resolution
    n_fft = 2048
    hop_length = 256
    
    # Compute STFT
    D = librosa.stft(y_clean, n_fft=n_fft, hop_length=hop_length)
    D_mag = np.abs(D)
    D_db = librosa.amplitude_to_db(D_mag, ref=np.max)
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # Linear frequency scale (good for seeing harmonics)
    img1 = librosa.display.specshow(D_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
    axes[0].set_ylim([0, fmax])
    axes[0].set_title(f'Cleaned Signal Spectrogram (0-{fmax} Hz) - Linear Scale', 
                      fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Frequency (Hz)')
    plt.colorbar(img1, ax=axes[0], format='%+2.0f dB')
    
    # Log frequency scale (good for seeing low-frequency detail)
    img2 = librosa.display.specshow(D_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='log', ax=axes[1], cmap='magma')
    axes[1].set_ylim([50, fmax])
    axes[1].set_title(f'Cleaned Signal Spectrogram (50-{fmax} Hz) - Log Scale',
                      fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Frequency (Hz)')
    axes[1].set_xlabel('Time (s)')
    plt.colorbar(img2, ax=axes[1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

# Plot detailed view
plot_detailed_spectrograms(y_clean, sr, fmax=8000)

## Parameter Tuning Guide

### Alpha (Over-subtraction factor)
- **Too low (< 1.5)**: Residual noise remains, air sound still audible
- **Optimal (1.5-2.5)**: Clean process sound with minimal artifacts
- **Too high (> 3.0)**: Musical noise artifacts, distorted process sounds

### Beta (Spectral floor)
- **Too low (< 0.005)**: Musical noise artifacts (random chirps)
- **Optimal (0.01-0.02)**: Smooth spectrum, no artifacts
- **Too high (> 0.05)**: Residual noise, less effective subtraction

### FFT Size
- **Smaller (1024-2048)**: Better time resolution, useful for transients
- **Larger (4096-8192)**: Better frequency resolution, smoother subtraction

### Recommendation
Start with: `alpha=2.0, beta=0.01, n_fft=4096`

Then adjust based on results:
1. Listen to the output
2. Check the spectrograms for remaining noise or artifacts
3. Adjust alpha first, then beta if needed